# Plain／residual 對照

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/03-comparison/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

### 可選：40 步學習實驗

跑完下面的完整實驗後，另開一個 code cell 執行：

```python
!python scripts/run_learning_extensions.py --section 03-comparison
from IPython.display import SVG, display
display(SVG(filename='artifacts/runs/learning/03-comparison/learning.svg'))
```

資料、optimizer 與結果的限制見網頁；這是另外的補充實驗，不取代下面的完整實驗。

In [1]:
"""Budget-matched small plain/residual comparison, not a ResNet benchmark."""
import copy
from time import perf_counter
import torch
from torch import nn


class Block(nn.Module):
    def __init__(self, residual):
        super().__init__()
        self.residual = residual
        self.branch = nn.Sequential(nn.Conv2d(4, 4, 3, padding=1, bias=False), nn.ReLU(),
                                    nn.Conv2d(4, 4, 3, padding=1, bias=False))

    def forward(self, x):
        correction = self.branch(x)
        return x + correction if self.residual else correction


class Classifier(nn.Module):
    def __init__(self, residual):
        super().__init__()
        self.stem = nn.Conv2d(3, 4, 3, padding=1)
        self.blocks = nn.Sequential(*[Block(residual) for _ in range(3)])
        self.head = nn.Linear(4, 2)

    def forward(self, x):
        x = self.blocks(nn.functional.relu(self.stem(x)))
        return self.head(x.mean((2, 3)))


def data(n, shift):
    images, labels = torch.zeros(n, 3, 16, 16), torch.arange(n) % 2
    for i in range(n):
        # left follows i // 2, not the label i % 2, so each red/blue pair shares one position
        left = 2 + ((i // 2) + shift) % 4
        top = 3 + shift
        images[i, 0 if labels[i] == 0 else 2, top:top + 8, left:left + 8] = 1
    shapes = images.amax(1)  # [n,16,16]: where each square is, color ignored
    tops, lefts = shapes.amax(2).argmax(1), shapes.amax(1).argmax(1)  # first lit row, first lit column
    corners = torch.stack((tops, lefts), 1)  # [n,2]: (top, left) of each square
    # Red and blue use the same (top, left) positions equally often (equal sorted lists); with every
    # square 8x8 (drawn above), only color separates the classes.
    assert sorted(corners[labels == 0].tolist()) == sorted(corners[labels == 1].tolist()), corners.tolist()
    return images, labels


def count_per_image(model, images):
    """Multiply-accumulates and shortcut additions per image, counted by forward hooks."""
    macs, shortcut_adds = [], []

    def hook(layer, inputs, output):
        values = output[0].numel()  # values this layer outputs for image 0
        if isinstance(layer, nn.Conv2d):  # each output value needs in_channels x 3 x 3 MACs
            macs.append(values * layer.in_channels * layer.kernel_size[0] ** 2)
        elif isinstance(layer, nn.Linear):  # each output value needs in_features MACs
            macs.append(values * layer.in_features)
        elif isinstance(layer, Block) and layer.residual:  # x + F(x): one add per output value
            shortcut_adds.append(values)

    handles = [layer.register_forward_hook(hook) for layer in model.modules()
               if isinstance(layer, (nn.Conv2d, nn.Linear, Block))]
    with torch.no_grad():
        model(images)
    for handle in handles:
        handle.remove()
    return sum(macs), sum(shortcut_adds)


def train_step(model, optimizer, images, labels):
    """One SGD update on the whole batch; returns the pre-update loss and stem-weight gradient L2 norm."""
    optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    loss = nn.functional.cross_entropy(logits, labels)
    loss.backward()
    grad_norm = model.stem.weight.grad.norm().item()
    assert grad_norm > 0 and torch.isfinite(loss)
    assert torch.isfinite(model.stem.weight.grad).all()
    optimizer.step()
    return loss.item(), grad_norm


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    train_x, train_y = data(8, 0)
    val_x, val_y = data(4, 2)
    assert not any(torch.equal(t, v) for t in train_x for v in val_x)
    plain, residual = Classifier(False), Classifier(True)
    residual.load_state_dict(plain.state_dict())
    for p, r in zip(plain.parameters(), residual.parameters()):
        assert torch.equal(p, r)
    for name, model in (("plain", plain), ("residual", residual)):
        params = sum(p.numel() for p in model.parameters())
        assert params == 986
        macs, shortcut_adds = count_per_image(model, train_x)
        assert macs == 248840
        assert shortcut_adds == (3072 if name == "residual" else 0)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
        before = model.stem.weight.detach().clone()
        # Untimed warm-up: the same step once on a throwaway copy, so the step's first-call
        # start-up work is not charged to whichever model happens to be timed first.
        throwaway = copy.deepcopy(model)
        train_step(throwaway, torch.optim.SGD(throwaway.parameters(), lr=0.1), train_x, train_y)
        assert torch.equal(before, model.stem.weight)  # the warm-up left the real model untouched
        start = perf_counter()
        model.train()
        for step in range(3):
            loss, grad_norm = train_step(model, optimizer, train_x, train_y)
            print(f"{name} step={step}, loss={loss:.4f}, stem_grad_norm={grad_norm:.6f}")
        elapsed = perf_counter() - start
        assert not torch.equal(before, model.stem.weight)
        model.eval()
        with torch.no_grad():
            acc = (model(val_x).argmax(1) == val_y).float().mean().item()
        print(f"{name}: params={params}, MACs/image={macs}, shortcut_adds/image={shortcut_adds}, "
              f"validation_accuracy={acc:.2f}, 3_step_seconds={elapsed:.4f}")
    print("Same initial weights/data/optimizer/steps; 3 steps and 4 validation images do not rank architectures.")


if __name__ == "__main__":
    main()


plain step=0, loss=0.6938, stem_grad_norm=0.000984
plain step=1, loss=0.6937, stem_grad_norm=0.001001
plain step=2, loss=0.6936, stem_grad_norm=0.001011
plain: params=986, MACs/image=248840, shortcut_adds/image=0, validation_accuracy=0.50, 3_step_seconds=0.0111
residual step=0, loss=0.6921, stem_grad_norm=0.146701
residual step=1, loss=0.6888, stem_grad_norm=0.147590
residual step=2, loss=0.6855, stem_grad_norm=0.146128
residual: params=986, MACs/image=248840, shortcut_adds/image=3072, validation_accuracy=0.50, 3_step_seconds=0.0104
Same initial weights/data/optimizer/steps; 3 steps and 4 validation images do not rank architectures.
